# Phase 11: Smith-Waterman CUDA Prototype


This notebook validates the Phase 11 Smith-Waterman CUDA prototype. The GPU implementation performs local alignment with wavefront parallelism, zero-reset dynamic programming cells, and maximum-score reduction.

In [ ]:
!nvidia-smi
!nvcc --version


In [ ]:
import os
print("Current working directory:", os.getcwd())


Compile the CPU reference and GPU implementation.

In [ ]:
!g++ src/smith_waterman_cpu.cpp \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o smith_waterman_cpu

!nvcc src/smith_waterman_gpu.cu \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o smith_waterman_gpu


Compile and run the GPU validation tests.

In [ ]:
!g++ tests/test_smith_waterman_gpu.cpp \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o test_smith_waterman_gpu


In [ ]:
!./test_smith_waterman_gpu


Generate a small fixed-length synthetic dataset.

In [ ]:
!python scripts/generate_synthetic_dataset.py \
  --num-pairs 100 \
  --sequence-length 32 \
  --output data/synthetic/synthetic_pairs_32.txt \
  --seed 42


Run the GPU implementation.

In [ ]:
!mkdir -p results/smith_waterman

!./smith_waterman_gpu \
  data/synthetic/synthetic_pairs_32.txt \
  results/smith_waterman/smith_waterman_gpu_results.csv \
  --repetitions 5 \
  --implementation wavefront


Run the benchmark and generate charts.

In [ ]:
!python benchmarks/run_smith_waterman_gpu_benchmark.py


In [ ]:
!python scripts/plot_smith_waterman_gpu_benchmark.py


Display benchmark results.

In [ ]:
import pandas as pd
df = pd.read_csv("benchmarks/smith_waterman_gpu_benchmark_results.csv")
df


Display generated charts.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

chart_directory = Path("assets/benchmark_charts/smith_waterman_gpu")
for chart_path in sorted(chart_directory.glob("*.png")):
    print(chart_path)
    display(Image(filename=str(chart_path)))
